# HiGAN + Generated-Style SupCon
Enable a Kaggle GPU and Internet, then run all cells.

In [ ]:
# Chỉ cần sửa cell này trước mỗi lần chạy.
REPO_URL = 'https://github.com/<owner>/HiGAN-Contrastive.git'
GIT_BRANCH = 'main'
DATASET_FORMAT = 'higan_hdf5'  # higan_hdf5 or iam_raw
DATASET_ROOT = '/kaggle/input/<dataset-slug>/higan_hdf5'
RUN_MODE = 'contrastive'  # baseline hoặc contrastive
RESUME_PATH = ''  # đường dẫn last.pt của phiên trước, hoặc để trống
MAX_STEPS = 0  # 0 = không giới hạn; số dương = dừng đúng global step đó
CONTRASTIVE_START_STEP = 28841  # step đầu tiên có SupCon
CONTRASTIVE_RAMP_STEPS = 28840  # 0/1 = bật ngay lambda_con tối đa
RECOGNIZER_CHECKPOINT = ''  # recognizer độc lập, hoặc để trống
SKIP_HWD = True  # True: chỉ FID/KID; False: tính thêm HWD
HWD_CHECKPOINT = ''  # local path hoặc hf://owner/repo/file.pth@revision; trống = nguồn chính thức
MAX_HOURS = 11.5
EVAL_MAX_SAMPLES = 0  # 0 = toàn bộ test.jsonl; số dương = giới hạn số ảnh duy nhất
FID_KID_PROTOCOL = 'higan'  # higan = setting gốc; torchvision = evaluator cũ của repo này
EVAL_SEED = 123456  # cùng noise eval cho baseline và contrastive


In [ ]:
REPO_DIR = '/kaggle/working/HiGAN-Contrastive'
!if [ -d "{REPO_DIR}/.git" ]; then git -C "{REPO_DIR}" checkout "{GIT_BRANCH}" && git -C "{REPO_DIR}" pull --ff-only origin "{GIT_BRANCH}"; else git clone --branch "{GIT_BRANCH}" "{REPO_URL}" "{REPO_DIR}"; fi
!git -C "{REPO_DIR}" rev-parse HEAD
%cd {REPO_DIR}


In [ ]:
MANIFEST_ROOT = '/kaggle/working/data_higan_hdf5' if DATASET_FORMAT == 'higan_hdf5' else '/kaggle/working/data'
!bash scripts/kaggle_setup.sh "{DATASET_ROOT}" "{DATASET_FORMAT}" "{MANIFEST_ROOT}"


In [ ]:
CONFIG = f'configs/kaggle_{RUN_MODE}.yaml'
SMOKE_OUTPUT = f'/kaggle/working/smoke/{RUN_MODE}'
!python train.py --config "{CONFIG}" --dataset-root "{DATASET_ROOT}" --dataset-format "{DATASET_FORMAT}" --manifest-root "{MANIFEST_ROOT}" --output "{SMOKE_OUTPUT}" --smoke-test --contrastive-start-step "{CONTRASTIVE_START_STEP}" --contrastive-ramp-steps "{CONTRASTIVE_RAMP_STEPS}"
from IPython.display import Image, display
display(Image(filename=f'{SMOKE_OUTPUT}/previews/step_00000001.png'))


In [ ]:
import shlex
RUN_MODE = RUN_MODE.strip().lower()
assert RUN_MODE in {'baseline', 'contrastive'}, repr(RUN_MODE)
FID_KID_PROTOCOL = FID_KID_PROTOCOL.strip().lower()
assert FID_KID_PROTOCOL in {'higan', 'torchvision'}, repr(FID_KID_PROTOCOL)
run_args = [
    'bash', 'scripts/kaggle_run.sh',
    '--dataset-root', DATASET_ROOT,
    '--dataset-format', DATASET_FORMAT,
    '--manifest-root', MANIFEST_ROOT,
    '--run-mode', RUN_MODE,
    '--max-hours', str(MAX_HOURS),
    '--max-steps', str(MAX_STEPS),
    '--contrastive-start-step', str(CONTRASTIVE_START_STEP),
    '--contrastive-ramp-steps', str(CONTRASTIVE_RAMP_STEPS),
    '--eval-max-samples', str(EVAL_MAX_SAMPLES),
    '--fid-kid-protocol', FID_KID_PROTOCOL,
    '--eval-seed', str(EVAL_SEED),
]
if RESUME_PATH:
    run_args += ['--resume', RESUME_PATH]
if RECOGNIZER_CHECKPOINT:
    run_args += ['--recognizer-checkpoint', RECOGNIZER_CHECKPOINT]
if HWD_CHECKPOINT:
    run_args += ['--hwd-checkpoint', HWD_CHECKPOINT]
if SKIP_HWD:
    run_args += ['--skip-hwd']
run_command = shlex.join(run_args)
print(run_command)
!{run_command}
from pathlib import Path
preview_files = sorted((Path('/kaggle/working/outputs') / RUN_MODE / 'previews').glob('*.png'))
if preview_files:
    display(Image(filename=str(preview_files[-1])))
evaluation_path = Path('/kaggle/working/outputs') / RUN_MODE / 'evaluation.json'
if evaluation_path.exists():
    print(evaluation_path.read_text())
print('/kaggle/working/run_bundle.tar.gz')
